# Eurovision Song Lyrics — Semantic Feature Scoring

Pipeline:
1. Load dataset
2. Inspect columns
3. Score each song on a set of semantic features using zero-shot NLI (multilingual)
4. Save enriched dataset

In [1]:
!pip install -q transformers accelerate sentencepiece pandas

## 1. Load dataset

File is in Google Drive at `Colab Notebooks/eurovision-lyrics-2025.json`. Mounting Drive below (you'll be asked to authorize once).

In [2]:
from google.colab import drive

drive.mount("/content/drive")

data_path = "/content/drive/MyDrive/Colab Notebooks/eurovision-lyrics-2025.json"
print("Using file:", data_path)

Mounted at /content/drive
Using file: /content/drive/MyDrive/Colab Notebooks/eurovision-lyrics-2025.json


## 2. Parse and inspect

In [3]:
import json
import pandas as pd

with open(data_path, encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame.from_dict(raw, orient="index").reset_index(drop=True)
print(df.shape)
print(df.columns.tolist())
df.head()

(1795, 14)
['#', 'Country', '#.1', 'Artist', 'Song', 'Language', 'Place', 'Score', 'Eurovision Number', 'Year', 'Host Country', 'Host City', 'Lyrics', 'Lyrics translation']


,#,Country,#.1,Artist,Song,Language,Place,Score,Eurovision Number,Year,Host Country,Host City,Lyrics,Lyrics translation
0,1,Netherlands,1,Jetty Paerl,De vogels van Holland,Dutch,-,-,1,1956,Switzerland,Lugano,De vogels van Holland zijn zo muzikaal\nZe ler...,The birds of Holland are so musical\nThey alre...
1,2,Switzerland,1,Lys Assia,Das alte Karussell,German,-,-,1,1956,Switzerland,Lugano,Das alte Karussell\nDas geht nicht mehr so sch...,The old carousel\nIt doesn't go as fast anymor...
2,3,Belgium,1,Fud Leclerc,Messieurs les noyés de la Seine,French,-,-,1,1956,Switzerland,Lugano,Messieurs les noyés de la Seine\nOuvrez-moi le...,Ye drowned men of the river Seine (1)\nOpen th...
3,4,Germany (West),1,Walter Andreas Schwarz,Im Wartesaal zum großen Glück,German,-,-,1,1956,Switzerland,Lugano,Es gibt einen Hafen\nDa fährt kaum ein Schiff\...,There is a harbour\nWhere hardly any ship leav...
4,5,France,1,Mathé Altéry,Le temps perdu,French,-,-,1,1956,Switzerland,Lugano,"Chante, carillon\nLe chant du temps perdu\nCha...","Sing, carillon\nThe song of lost time\nSing yo..."


## 3. Semantic feature scoring

Uses a multilingual NLI model as a zero-shot classifier: for each song and each feature,
it scores how much the lyrics text "entails" the label (e.g. "this text is about love").
No training data needed — works directly on raw lyrics in any language.

Scoring runs in batches (fed as a generator so the GPU is used efficiently instead of
one-by-one calls) and checkpoints progress to Google Drive every `SAVE_EVERY` rows.
If the runtime disconnects or the cell is interrupted, just rerun it — it resumes from
where it left off instead of starting over.

**Set `LYRICS_COL` below to match your dataset's actual lyrics column name** (check the printed column list above).

In [4]:
LYRICS_COL = "Lyrics"  # <-- change if needed based on df.columns above

# Features to score — edit this list to whatever dimensions you care about
FEATURES = [
    "love",
    "hope",
    "heartbreak",
    "party / celebration",
    "war or conflict",
    "nostalgia",
    "empowerment",
    "loneliness",
    "nature",
    "spirituality",
]

In [5]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device,
)

Using GPU


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [6]:
import os
from tqdm.auto import tqdm

# Quick sanity check on one song before running the full batch
sample = df[LYRICS_COL].iloc[0]
print(classifier(sample, candidate_labels=FEATURES, multi_label=True))

{'sequence': 'De vogels van Holland zijn zo muzikaal\nZe leren in hun prille jeugd al tierelieren\nDe merel, de lijster en de nachtegaal\nOm zo de lente in Holland goed te kunnen vieren\n\nHet is geen wonder, want nergens\nZijn de plassen zo blauw\nAls in Holland - mijnheer\nAls in Holland - mevrouw\nHet is geen wonder, want nergens\nIs het gras zo vol dauw\nZijn de meisjes zo lief\nZijn de meisjes zo trouw\nEn daarom zijn de vogels hier allemaal\nZo muzikaal, zo muzikaal, zo muzikaal\n\nDe hele wereld door heb ik vogels horen zingen\nIn het zuiden, in het westen, in het noorden\nIn vele verre landen heb ik vogels horen zingen\nZij zingen kleine liedjes zonder woorden\n\nDe Franse vogels zingen "tudeludelu"\nJapanse vogels zingen "tudeludelu"\nChinese vogels zingen "tudeludelu"\nMaar de vogels zingen nergens\nZo gelukkig en blij\nAls in Holland in het voorjaar in de wei\n\nDe vogels van Holland zijn zo muzikaal\nZe leren in hun prille jeugd al tierelieren\nDe merel, de lijster en de na

In [7]:
CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/eurovision_scores_checkpoint.csv"
SAVE_EVERY = 25
BATCH_SIZE = 16

if os.path.exists(CHECKPOINT_PATH):
    scores_df = pd.read_csv(CHECKPOINT_PATH)
    start_idx = len(scores_df)
    print(f"Resuming from row {start_idx}/{len(df)}")
else:
    scores_df = pd.DataFrame(columns=FEATURES)
    start_idx = 0

remaining_lyrics = df[LYRICS_COL].iloc[start_idx:].fillna("").tolist()

def lyrics_gen():
    for text in remaining_lyrics:
        yield text if text.strip() else "no lyrics"

pending = []

if remaining_lyrics:
    result_iter = classifier(lyrics_gen(), candidate_labels=FEATURES, multi_label=True, batch_size=BATCH_SIZE)
    for result in tqdm(result_iter, total=len(remaining_lyrics)):
        pending.append(dict(zip(result["labels"], result["scores"])))
        if len(pending) >= SAVE_EVERY:
            scores_df = pd.concat([scores_df, pd.DataFrame(pending)], ignore_index=True)
            scores_df.to_csv(CHECKPOINT_PATH, index=False)
            pending = []
    if pending:
        scores_df = pd.concat([scores_df, pd.DataFrame(pending)], ignore_index=True)
        scores_df.to_csv(CHECKPOINT_PATH, index=False)

print("Scoring complete:", len(scores_df), "rows")

  0%|          | 0/1795 [00:00<?, ?it/s]

/tmp/ipykernel_507/1694219136.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores_df = pd.concat([scores_df, pd.DataFrame(pending)], ignore_index=True)


Scoring complete: 1795 rows


In [8]:
enriched_df = pd.concat([df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)
enriched_df.head()

,#,Country,#.1,Artist,Song,Language,Place,Score,Eurovision Number,Year,...,love,hope,heartbreak,party / celebration,war or conflict,nostalgia,empowerment,loneliness,nature,spirituality
0,1,Netherlands,1,Jetty Paerl,De vogels van Holland,Dutch,-,-,1,1956,...,0.911604,0.865110,0.002895,0.843895,0.232223,0.389314,0.760295,0.060752,0.938315,0.564339
1,2,Switzerland,1,Lys Assia,Das alte Karussell,German,-,-,1,1956,...,0.784690,0.969164,0.494385,0.955232,0.721743,0.965029,0.816699,0.677207,0.882798,0.470737
2,3,Belgium,1,Fud Leclerc,Messieurs les noyés de la Seine,French,-,-,1,1956,...,0.003594,0.028008,0.300617,0.431944,0.693680,0.763616,0.053999,0.784381,0.809464,0.134591
3,4,Germany (West),1,Walter Andreas Schwarz,Im Wartesaal zum großen Glück,German,-,-,1,1956,...,0.468183,0.526404,0.014064,0.586286,0.369998,0.894005,0.717507,0.540196,0.458591,0.470250
4,5,France,1,Mathé Altéry,Le temps perdu,French,-,-,1,1956,...,0.991954,0.960100,0.969584,0.863334,0.104021,0.985000,0.626867,0.844537,0.447098,0.584864


## 4. Save

In [9]:
output_path = "/content/drive/MyDrive/Colab Notebooks/eurovision_enriched.csv"
enriched_df.to_csv(output_path, index=False)
print("Saved to", output_path)

Saved to /content/drive/MyDrive/Colab Notebooks/eurovision_enriched.csv
